**Run order:** Select the Docker Jupyter kernel and run the cells from top to bottom.
Use the Docker Jupyter server at `http://localhost:8888`. Credentials are read from the mounted project `.env`; no manual key input is required.
Spark CPU and memory settings use their defaults.


In [1]:
# Install the small .env reader in this notebook kernel only if needed.
import importlib.util

if importlib.util.find_spec("dotenv") is None:
    get_ipython().run_line_magic("pip", "install -q python-dotenv")


In [1]:
from pathlib import Path
from dotenv import dotenv_values
from pyspark.sql import SparkSession, functions as F, Window

# This project is mounted inside spark-notebook at /home/jovyan/work.
env_path = Path("/home/jovyan/work/.env")
if not env_path.is_file():
    raise FileNotFoundError(
        "Project .env is missing. Select the Docker Jupyter server kernel "
        "and check the /home/jovyan/work volume mount."
    )

settings = dotenv_values(env_path)
access_key = settings.get("MINIO_ACCESS_KEY") or settings.get("MINIO_ROOT_USER")
secret_key = settings.get("MINIO_SECRET_KEY") or settings.get("MINIO_ROOT_PASSWORD")
if not access_key or not access_key.strip() or not secret_key or not secret_key.strip():
    raise ValueError("MINIO_ACCESS_KEY and MINIO_SECRET_KEY must be nonempty in .env.")

# Use the MinIO alias on spark-net, not localhost inside the notebook container.
s3a_options = {
    "fs.s3a.endpoint": "http://100.72.238.14:19000",
    "fs.s3a.access.key": access_key,
    "fs.s3a.secret.key": secret_key,
    "fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    "fs.s3a.path.style.access": "true",
    "fs.s3a.connection.ssl.enabled": "false",
    "fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
}

spark = (
    SparkSession.builder
    .appName("batch-analysis")
    # Sort shuffle output instead of opening a compression buffer per destination.
    .config("spark.shuffle.sort.bypassMergeThreshold", "0")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262",
    )
    .getOrCreate()
)

# Apply MinIO settings to the context that will actually perform the read.
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
for key, value in s3a_options.items():
    hadoop_conf.set(key, value)
# The kernel may already have an S3A client from a previous read.
hadoop_conf.setBoolean("fs.s3a.impl.disable.cache", True)

print("Spark initialized:", spark.version)
print("MinIO credentials loaded from .env; values are hidden.")


Spark initialized: 3.5.3
MinIO credentials loaded from .env; values are hidden.


In [2]:
# Đọc dữ liệu Bronze từ MinIO

# Ngày và giờ trong các thống kê bên dưới đều theo UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

BRONZE_PATH = "s3a://bronze/events/*.jsonl.gz"

raw = (
    spark.read
    .option("mode", "FAILFAST")
    .json(BRONZE_PATH)
)

raw.printSchema()

root
 |-- album_metadata: struct (nullable = true)
 |    |-- playcount: string (nullable = true)
 |    |-- tracks: struct (nullable = true)
 |    |    |-- track: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- @attr: struct (nullable = true)
 |    |    |    |    |    |-- rank: long (nullable = true)
 |    |    |    |    |-- artist: struct (nullable = true)
 |    |    |    |    |    |-- mbid: string (nullable = true)
 |    |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |    |    |-- url: string (nullable = true)
 |    |    |    |    |-- duration: long (nullable = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |    |-- streamable: struct (nullable = true)
 |    |    |    |    |    |-- #text: string (nullable = true)
 |    |    |    |    |    |-- fulltrack: string (nullable = true)
 |    |    |    |    |-- url: string (nullable = true)
 |    |-- userplaycount: string (nullable

In [ ]:
# Để cập nhật danh sách file, chạy lại cell đọc Bronze phía trên.
print("tổng dòng:", raw.count())
print("số file:", raw.select("_metadata.file_path").distinct().count())

raw.groupBy("source").count().orderBy(F.desc("count")).show(truncate=False)

(raw.groupBy(F.lower("user_profile.name").alias("user"))
    .count().orderBy(F.desc("count")).show(20, truncate=False))

## 1. Từ `raw` đến bảng lượt nghe `plays`

Chạy từ đây khi kernel đã có `raw`. Mỗi dòng của `plays` là một scrobble Last.fm có `event_id`, user và thời điểm nghe hợp lệ. `uts` là Unix giây. Giờ, thứ, ngày, tháng và ranh giới tuần đều tính theo UTC.

Lấy tên bài và nghệ sĩ từ chính scrobble. Dùng `user_profile.name` để nhận diện user. Các bộ đếm `playcount` trong metadata là số tích lũy nên không tham gia phép đếm lượt nghe.

In [3]:
import csv
import sys
from pyspark import StorageLevel
from pyspark.sql import Window, functions as F


def show_all(frame):
    """In tiêu đề và từng dòng của bảng theo thứ tự hiện tại."""
    writer = csv.writer(sys.stdout, delimiter="\t", lineterminator="\n")
    writer.writerow(frame.columns)
    # Spark định dạng cả timestamp theo múi giờ UTC đã cấu hình trong notebook.
    formatted = frame.select(*(F.col(c).cast("string") for c in frame.columns))
    for row in formatted.toLocalIterator():
        writer.writerow("NULL" if value is None else value for value in row)


spark.conf.set("spark.sql.session.timeZone", "UTC")
events = (
    raw.where(F.col("source") == "lastfm")
    .withColumn("uts", F.expr("try_cast(scrobble_event.date.uts AS BIGINT)"))
    .withColumn("user", F.lower(F.trim("user_profile.name")))
)
# Loại UTS thiếu, sai kiểu hoặc ngoài khoảng 1970–9999 trước khi khử trùng.
eligible = events.where(
    F.col("uts").isNotNull() & F.col("uts").between(0, 253402300799)
    & (F.length(F.trim("event_id")) > 0) & (F.length("user") > 0)
)
plays = (
    eligible.select("event_id", "user", "uts",
                    F.col("scrobble_event.name").alias("track"),
                    F.col("scrobble_event.artist.name").alias("artist"))
    .dropDuplicates(["event_id"])
    .withColumn("ts_utc", F.timestamp_seconds("uts"))
    .withColumn("ngay_utc", F.to_date("ts_utc"))
    .withColumn("gio_utc", F.hour("ts_utc"))
    .withColumn("thu_iso", F.weekday("ts_utc") + 1)
    .withColumn("thang", F.month("ts_utc"))
    .withColumn("cuoi_tuan", (F.col("thu_iso") >= 6).cast("int"))
    .withColumn("tuan_utc", F.to_date(F.date_trunc("week", "ts_utc")))
    .persist(StorageLevel.DISK_ONLY)  # Reuse the deduplicated plays across analyses.
)
n_raw = events.count()
n_valid = eligible.count()
n_plays = plays.count()
print(f"{n_raw} dòng Last.fm | {n_raw - n_valid} dòng thiếu/sai khóa hoặc UTS | "
      f"{n_valid - n_plays} dòng trùng | {n_plays} lượt dùng để phân tích")
show_all(plays.orderBy("uts", "event_id"))

271072 dòng Last.fm | 0 dòng thiếu/sai khóa hoặc UTS | 6068 dòng trùng | 265004 lượt dùng để phân tích
event_id	user	uts	track	artist	ts_utc	ngay_utc	gio_utc	thu_iso	thang	cuoi_tuan	tuan_utc
lastfm:9aaed45ad63bfc3075a6683cdd620034ef489f22788d402169b3f6fc73cdf710	t28foever	1795449600	Banquet	Bloc Party	2026-11-23 16:00:00	2026-11-23	16	1	11	0	2026-11-23
lastfm:9f9cbd8864b42da59f63f199cfdf33ab2b9224f8d4894fba04660f0f69f9eec0	ftriples6x	1795449600	cliche	2hollis	2026-11-23 16:00:00	2026-11-23	16	1	11	0	2026-11-23
lastfm:fd46d999b3f5b4f944edf6be5d65ec3c03f2a1715f0ac7241b12a0d9fee1c1a1	cinnymoncinder	1795449600	Unknown Track 4	David Javelosa	2026-11-23 16:00:00	2026-11-23	16	1	11	0	2026-11-23
lastfm:044da2099e237569009a21994dd558603f95b8272c79b26480185a9a135db1a5	sibonforever	1795449601	Let's Put Out The Lights - Remastered 2023	Corry Brokken	2026-11-23 16:00:01	2026-11-23	16	1	11	0	2026-11-23
lastfm:2cea217ce1915881bede205131d7bdd754d6c8557a81b47e33bf62cf1dc76e29	sylvaincourtoux	1795449601

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



lastfm:16f6a79d8040cd5d8107cef3312e82e5d01a0fd4773a646b69649b8e2c601657	akeripper	1795469923	NEVER ENOUGH	Turnstile	2026-11-23 21:38:43	2026-11-23	21	1	11	0	2026-11-23
lastfm:7f4261fbd23c1e59e20ebb08753f775fcc7e7876af0bd4be67f2cc56cc6976dc	paco10mz	1795469924	Vale La Pena	Los Tigres del Norte	2026-11-23 21:38:44	2026-11-23	21	1	11	0	2026-11-23
lastfm:31e1ca9c993bf9a492aba0d28ca1b6e30e7f73fbf2bb032839507404f35b97e4	nadirinabottle	1795469926	You're My First Love (Head Over Heels) (Full Version, Musikladen Eurotops, 1987)	New Romance	2026-11-23 21:38:46	2026-11-23	21	1	11	0	2026-11-23
lastfm:c38893fbc2a8f39dca9d444748a84f9eae74286c24370bdd1b2d29d924323c4a	raphahead	1795469926	Betelgeuse	King Creosote	2026-11-23 21:38:46	2026-11-23	21	1	11	0	2026-11-23
lastfm:2ab11b5d84f480a73883f2443c1063a9614a020d4863f671c4bc0aa6071c782d	tr4czyk	1795469927	Articulate (Im Going So Far Into You)	The I.L.Y's	2026-11-23 21:38:47	2026-11-23	21	1	11	0	2026-11-23
lastfm:4040740ebe17de161d6486d0b044d599afa00d0d6

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



lastfm:3c261a9a5c6230f31e05569cfeed5b274553eb87baf9e475eb0d081b51d5e197	ygivo	1795492118	Kind israeli man gives 51 kebabs to a local mosque.	‘Nished SECA	2026-11-24 03:48:38	2026-11-24	3	2	11	0	2026-11-23
lastfm:c311499f1a0f84dfb21446a2ab18782ee4e050c91b8d52cf58526ba2bf50a548	payert	1795492118	Hopeful Hearts	Sarah Slean	2026-11-24 03:48:38	2026-11-24	3	2	11	0	2026-11-23
lastfm:edcacd2bab97c52f1ec0c4b70c9cd787970f0432af8014a1cbed6b2028f20b82	politemoon13	1795492118	Eye of the Tiger	Survivor	2026-11-24 03:48:38	2026-11-24	3	2	11	0	2026-11-23
lastfm:fdb560baf9c07f92657b2fa92040f4d3f84aec84a1803ab5cb6d9b92203c1620	catonaleash	1795492119	Phantom	Atlus Sound Team	2026-11-24 03:48:39	2026-11-24	3	2	11	0	2026-11-23
lastfm:1f7f95b5ccb08288fe0ab1b9b7df722a00f3808626c2f47a07f83dc07e238ae2	twifip	1795492120	Butterfly	Sons of Sevilla	2026-11-24 03:48:40	2026-11-24	3	2	11	0	2026-11-23
lastfm:36a9a052e03850e483a71b0599f8c8635b30739d337c560021b4a268b894d662	tkeen88	1795492120	チャンセズ・アー	Clémentine	2026-

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



... (4 outputs left out)

lastfm:c6062688ecf8ab9bf12d7446324b24e15158d34299d5a22ace9d46f9c8d55d76	pcinspektor	1795707843	Fantastic Voyage	Lakeside	2026-11-26 15:44:03	2026-11-26	15	4	11	0	2026-11-23
lastfm:d270b48391a947949bd700e1a35a7ea0b9c39a2a40ba0e3894358e3787b68ac5	errorgasm	1795707843	ROMANCE FDP (ANOS 2000)	DJ Holanda	2026-11-26 15:44:03	2026-11-26	15	4	11	0	2026-11-23
lastfm:f2076fa55d79f28987d3be0df1f77c6bb2a3ffbb7108fa93e130e91e01a8e33b	l1r	1795707843	Obstacle 2	Interpol	2026-11-26 15:44:03	2026-11-26	15	4	11	0	2026-11-23
lastfm:2b57a7012b0b74040a131d9ccbb5fa3073da3f5d3645b7dbc68f1235b9bc65a8	zytrock	1795707844	Hypnotize	System of a Down	2026-11-26 15:44:04	2026-11-26	15	4	11	0	2026-11-23
lastfm:f5589647823c2bc62af888710665348339fae64fcf8ef1a2d1bfe1b0a78cef7d	nazpyro	1795707844	Anything Goes	Florida Georgia Line	2026-11-26 15:44:04	2026-11-26	15	4	11	0	2026-11-23
lastfm:4118850358494cd33d03e296bfca792f7e4f7cdbe2aebd9ad87561336da5f6f5	goldniggaa	1795707845	Role Play	Santino Le Saint	2026-11-26 15:44:05

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



lastfm:c2115a42ba7f06c4388f6386439882e2a1f15be2a48a208322f03f99a0a932b8	eadeleke1	1795733700	Wear It Like a Crown	Rebekka Karijord	2026-11-26 22:55:00	2026-11-26	22	4	11	0	2026-11-23
lastfm:0f138f102be9f24f2255df351caad52856dd9aa516fc4c88a90506d0404752cf	itz_dennisss	1795733702	Father Stretch My Hands Pt. 1	Kanye West	2026-11-26 22:55:02	2026-11-26	22	4	11	0	2026-11-23
lastfm:f232c31c311a41e48107aac70199732e773c8b28524bcfaa2edc9edfa4183d2b	land0n1	1795733702	Black Skinhead	Kanye West	2026-11-26 22:55:02	2026-11-26	22	4	11	0	2026-11-23
lastfm:6daf1a43c837b97e52e9282b0a965f226053f2a054404d0c5a0c80d1195d1091	xeshaa	1795733704	Сон	УБИЙЦЫ CRYSTAL	2026-11-26 22:55:04	2026-11-26	22	4	11	0	2026-11-23
lastfm:20f6cdb5775e21cb749e391f1e8787ed7bef2e25cab41dcc7fe66614173deec1	ven-de-tta	1795733705	Kat Lazer Eye Chain	bleood	2026-11-26 22:55:05	2026-11-26	22	4	11	0	2026-11-23
lastfm:9e31b6ea71043fbfe4d03b572364a9669259a18ac264da578dd18895158fcca7	sa1zig	1795733705	No Church In The Wild	Jaÿ-Z	2026-11

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



## 2. Mỗi user xuất hiện trong bao nhiêu ngày?

Đọc `so_ngay_co_nghe` cùng ngày đầu, ngày cuối và số ngày lịch giữa hai mốc (tính cả hai đầu). Ví dụ, 2 ngày có nghe trong khoảng 30 ngày là tín hiệu cần đối chiếu với nhật ký crawl trước khi kết luận về phiên hoặc xu hướng. Một user chỉ xuất hiện một ngày có thể đạt 100% trên khoảng quan sát rất ngắn. Scrobble riêng lẻ không cho biết một ngày trống là do user nghỉ nghe hay do thiếu dữ liệu.

In [4]:
coverage = (
    plays.groupBy("user")
    .agg(F.count("*").alias("so_luot"),
         F.countDistinct("ngay_utc").alias("so_ngay_co_nghe"),
         F.min("ngay_utc").alias("ngay_dau"), F.max("ngay_utc").alias("ngay_cuoi"))
    .withColumn("so_ngay_lich", F.datediff("ngay_cuoi", "ngay_dau") + 1)
    .withColumn("pct_ngay_co_nghe", F.round(100 * F.col("so_ngay_co_nghe") / F.col("so_ngay_lich"), 1))
)
show_all(coverage.orderBy("pct_ngay_co_nghe", "so_ngay_co_nghe", "user"))
n_users = coverage.count()
if n_users == 0:
    raise ValueError("Không có scrobble Last.fm hợp lệ. Kiểm tra nguồn, event_id, user và UTS ở bước 1.")

user	so_luot	so_ngay_co_nghe	ngay_dau	ngay_cuoi	so_ngay_lich	pct_ngay_co_nghe
hadesbae	34	2	2026-11-23	2026-11-27	5	40.0
antzus_ai	47	2	2026-11-24	2026-11-27	4	50.0
arsonscottbott	16	2	2026-11-24	2026-11-27	4	50.0
cthulhulicious	71	2	2026-11-23	2026-11-26	4	50.0
ducksdoormat	52	2	2026-11-23	2026-11-26	4	50.0
hueblitzz	10	2	2026-11-23	2026-11-26	4	50.0
kylekusch	145	2	2026-11-23	2026-11-26	4	50.0
lenrd	20	2	2026-11-23	2026-11-26	4	50.0
lipehlol	43	2	2026-11-24	2026-11-27	4	50.0
martingorewhore	33	2	2026-11-23	2026-11-26	4	50.0
popeli	7	2	2026-11-24	2026-11-27	4	50.0
re_sliced	47	2	2026-11-23	2026-11-26	4	50.0
silversonicaxel	107	2	2026-11-23	2026-11-26	4	50.0
thomasfez	94	2	2026-11-23	2026-11-26	4	50.0
virulentalli	22	2	2026-11-23	2026-11-26	4	50.0
ytssur	126	2	2026-11-23	2026-11-26	4	50.0
zan666	294	2	2026-11-23	2026-11-26	4	50.0
bblyzy	77	3	2026-11-23	2026-11-27	5	60.0
boingus724	25	3	2026-11-23	2026-11-27	5	60.0
burgerfacegg	34	3	2026-11-23	2026-11-27	5	60.0
cinnymoncinder	678	3	2026

## 3. Giờ UTC, thứ, tháng và cuối tuần

Tính tỷ lệ của từng user trước. Ví dụ, với 2 user, một người có 1/2 lượt lúc 20h và người kia có 9/10 lượt lúc 20h, kết quả ở 20h là (50% + 90%)/2 = **70%**. Mẫu số để lấy trung bình là **tất cả user** trong `plays`. User thiếu một khung giờ đóng góp 0 vào khung đó.

Mỗi bảng có đủ các nhóm lịch, tổng `share_pct` xấp xỉ 100% sau làm tròn. `so_luot` và `so_user_co_nghe` giúp kiểm tra kích thước mẫu. `thu_iso` chạy từ 1 (thứ Hai) đến 7 (Chủ nhật). `thang` là tháng trong năm, gộp các năm nếu có. Chỉ diễn giải mùa vụ khi dữ liệu phủ đủ các tháng.

Bảng cuối so sánh tỷ trọng lượt nghe của thứ Hai–thứ Sáu với thứ Bảy–Chủ nhật. Hai nhóm gồm 5 ngày và 2 ngày mỗi tuần, vì vậy tỷ trọng này chưa cho biết trung bình **mỗi ngày** cuối tuần nghe nhiều hay ít hơn.

In [13]:
def distribution(dimension, start, end):
    user_bins = (
        plays.groupBy("user", dimension).count()
        .join(coverage.select("user", "so_luot"), "user")
        .withColumn("user_share", F.col("count") / F.col("so_luot"))
    )
    totals = user_bins.groupBy(dimension).agg(
        F.sum("count").alias("so_luot"), F.count("*").alias("so_user_co_nghe"),
        F.round(100 * F.sum("user_share") / F.lit(n_users), 2).alias("share_pct"),
    )
    bins = spark.range(start, end + 1).select(F.col("id").alias(dimension))
    return bins.join(totals, dimension, "left").fillna(0).orderBy(dimension)

by_hour = distribution("gio_utc", 0, 23)
by_weekday = distribution("thu_iso", 1, 7)
by_month = distribution("thang", 1, 12)
by_day_type = distribution("cuoi_tuan", 0, 1).withColumn(
    "loai_ngay", F.when(F.col("cuoi_tuan") == 1, "Thứ Bảy–Chủ nhật").otherwise("Thứ Hai–thứ Sáu")
)

for table in (by_hour, by_weekday, by_month, by_day_type):
    show_all(table)

gio_utc	so_luot	so_user_co_nghe	share_pct
0	0	0	0.0
1	0	0	0.0
2	0	0	0.0
3	0	0	0.0
4	0	0	0.0
5	0	0	0.0
6	0	0	0.0
7	1	1	6.25
8	7	1	43.75
9	0	0	0.0
10	0	0	0.0
11	0	0	0.0
12	7	1	43.75
13	0	0	0.0
14	1	1	6.25
15	0	0	0.0
16	0	0	0.0
17	0	0	0.0
18	0	0	0.0
19	0	0	0.0
20	0	0	0.0
21	0	0	0.0
22	0	0	0.0
23	0	0	0.0
thu_iso	so_luot	so_user_co_nghe	share_pct
1	0	0	0.0
2	8	1	50.0
3	0	0	0.0
4	0	0	0.0
5	1	1	6.25
6	0	0	0.0
7	7	1	43.75
thang	so_luot	so_user_co_nghe	share_pct
1	0	0	0.0
2	0	0	0.0
3	0	0	0.0
4	0	0	0.0
5	0	0	0.0
6	0	0	0.0
7	0	0	0.0
8	0	0	0.0
9	0	0	0.0
10	0	0	0.0
11	16	1	100.0
12	0	0	0.0
cuoi_tuan	so_luot	so_user_co_nghe	share_pct	loai_ngay
0	9	1	56.25	Thứ Hai–thứ Sáu
1	7	1	43.75	Thứ Bảy–Chủ nhật


**Một ngày cuối tuần có nhiều lượt hơn một ngày thường hay không?** Với mỗi user có dữ liệu ở cả hai loại ngày, tính số lượt trung bình trên một **ngày có nghe** của từng loại. Cân hai mức này thành tỷ lệ sao cho tổng bằng 100%, rồi lấy trung bình đều giữa các user đủ điều kiện. Nếu `pct_cuoi_tuan` lớn hơn 50%, nhóm này có xu hướng nghe nhiều hơn trên một ngày cuối tuần có hoạt động. Ngày không có bản ghi vẫn chưa biết độ phủ crawl nên không đưa vào mẫu số. Nếu `so_user_so_sanh` bằng 0, chưa thể so sánh trên mẫu hiện tại.

In [14]:
per_active_day = (
    plays.groupBy("user", "cuoi_tuan")
    .agg((F.count("*") / F.countDistinct("ngay_utc")).alias("luot_moi_ngay_co_nghe"))
    .groupBy("user")
    .agg(F.max(F.when(F.col("cuoi_tuan") == 0, F.col("luot_moi_ngay_co_nghe"))).alias("ngay_thuong"),
         F.max(F.when(F.col("cuoi_tuan") == 1, F.col("luot_moi_ngay_co_nghe"))).alias("cuoi_tuan"))
    .dropna(subset=["ngay_thuong", "cuoi_tuan"])
    .withColumn("share_cuoi_tuan", F.col("cuoi_tuan") / (F.col("cuoi_tuan") + F.col("ngay_thuong")))
)
day_comparison = per_active_day.agg(
    F.count("*").alias("so_user_so_sanh"),
    F.round(100 * F.avg(1 - F.col("share_cuoi_tuan")), 2).alias("pct_ngay_thuong"),
    F.round(100 * F.avg("share_cuoi_tuan"), 2).alias("pct_cuoi_tuan"),
)
show_all(day_comparison)

so_user_so_sanh	pct_ngay_thuong	pct_cuoi_tuan
1	39.13	60.87


## 4. Chia phiên nghe theo user

Sắp lượt nghe theo `uts` trong từng user. Lượt đầu mở phiên 1. Khoảng cách **đúng 30 phút** vẫn thuộc cùng phiên, **trên 30 phút** mở phiên mới. Một phiên có thể đi qua nửa đêm. Khóa phiên là cặp `(user, phien)`.

`khoang_phut` đo thời gian giữa scrobble đầu và cuối phiên. Một phiên một lượt có giá trị 0. Đây là độ dài quan sát được. Nếu crawl bỏ sót lượt ở giữa, một phiên thực có thể bị tách làm hai. Đọc kết quả cùng `so_ngay_co_nghe` ở bước 2.

In [5]:
order_user = Window.partitionBy("user").orderBy("uts", "event_id")
running = order_user.rowsBetween(Window.unboundedPreceding, Window.currentRow)
session_plays = (
    plays.withColumn("gap_s", F.col("uts") - F.lag("uts").over(order_user))
    .withColumn("bat_dau", (F.col("gap_s").isNull() | (F.col("gap_s") > 1800)).cast("int"))
    .withColumn("phien", F.sum("bat_dau").over(running))
)
sessions = (
    session_plays.groupBy("user", "phien")
    .agg(F.min("ts_utc").alias("bat_dau_utc"), F.max("ts_utc").alias("ket_thuc_utc"),
         F.count("*").alias("luot_trong_phien"),
         F.round((F.max("uts") - F.min("uts")) / 60, 1).alias("khoang_phut"))
)
sessions_with_coverage = sessions.join(coverage.select("user", "so_ngay_co_nghe", "so_ngay_lich"), "user").orderBy(
    "user", "phien"
)
show_all(sessions_with_coverage)

user	phien	bat_dau_utc	ket_thuc_utc	luot_trong_phien	khoang_phut	so_ngay_co_nghe	so_ngay_lich
1100	1	2026-11-23 16:08:41	2026-11-23 19:47:38	27	219.0	4	5
1100	2	2026-11-23 20:24:27	2026-11-24 01:04:01	39	279.6	4	5
1100	3	2026-11-24 01:36:42	2026-11-24 02:46:19	14	69.6	4	5
1100	4	2026-11-24 03:36:36	2026-11-24 05:31:33	26	115.0	4	5
1100	5	2026-11-24 06:43:45	2026-11-24 09:47:17	37	183.5	4	5
1100	6	2026-11-24 10:43:34	2026-11-24 11:21:09	10	37.6	4	5
1100	7	2026-11-24 12:29:56	2026-11-24 13:18:01	11	48.1	4	5
1100	8	2026-11-24 14:01:38	2026-11-24 15:28:52	17	87.2	4	5
1100	9	2026-11-26 03:02:20	2026-11-26 03:29:06	6	26.8	4	5
1100	10	2026-11-26 04:04:30	2026-11-26 05:54:24	24	109.9	4	5
1100	11	2026-11-26 09:00:47	2026-11-26 13:14:37	57	253.8	4	5
1100	12	2026-11-26 23:14:13	2026-11-27 06:55:16	100	461.1	4	5
40belowsummer	1	2026-11-23 16:02:30	2026-11-23 16:23:43	7	21.2	4	5
40belowsummer	2	2026-11-23 16:55:51	2026-11-23 23:01:38	98	365.8	4	5
40belowsummer	3	2026-11-24 06:35:33	2026-11-24 07:09

## 5. Hoạt động tăng hay giảm qua các tuần UTC?

`tuan_utc` là ngày thứ Hai mở đầu tuần, gồm cả năm nên không gộp nhầm tuần giữa các năm. `share_pct` là tỷ trọng hoạt động của tuần trong toàn bộ mẫu, lấy trung bình đều theo user như bước 3.

Để đo thay đổi, chỉ so sánh những user **có lượt ở cả hai tuần lịch liên tiếp**. Với từng user, lấy tỷ trọng tuần này trừ tỷ trọng tuần trước rồi mới lấy trung bình. Dấu của `thay_doi_diem_pct` cho biết chiều thay đổi trong nhóm so sánh. Đọc kèm `so_user_so_sanh`. Giá trị trống nghĩa là chưa có cặp tuần đủ điều kiện. Hai tuần đầu/cuối được crawl một phần hoặc các tuần có lịch crawl khác nhau có thể làm sai lệch xu hướng.

In [6]:
user_weeks = (
    plays.groupBy("user", "tuan_utc").count()
    .join(coverage.select("user", "so_luot"), "user")
    .withColumn("user_share", F.col("count") / F.col("so_luot"))
)
weekly = user_weeks.groupBy("tuan_utc").agg(
    F.sum("count").alias("so_luot"), F.count("*").alias("so_user_co_nghe"),
    F.round(100 * F.sum("user_share") / F.lit(n_users), 2).alias("share_pct"),
)
w_week = Window.partitionBy("user").orderBy("tuan_utc")
paired = (
    user_weeks.withColumn("tuan_truoc", F.lag("tuan_utc").over(w_week))
    .withColumn("share_truoc", F.lag("user_share").over(w_week))
    .where(F.datediff("tuan_utc", "tuan_truoc") == 7)
    .groupBy("tuan_utc").agg(F.count("*").alias("so_user_so_sanh"),
        F.round(100 * F.avg(F.col("user_share") - F.col("share_truoc")), 2).alias("thay_doi_diem_pct"))
)
trend = weekly.join(paired, "tuan_utc", "left").fillna(0, subset=["so_user_so_sanh"])
show_all(trend.orderBy("tuan_utc"))

tuan_utc	so_luot	so_user_co_nghe	share_pct	so_user_so_sanh	thay_doi_diem_pct
2026-11-23	265004	808	100.0	0	NULL


## 6. Giờ UTC nào thường nghe tag gì?

Tag Last.fm là nhãn do cộng đồng gắn, có thể chỉ thể loại, tâm trạng hoặc chủ đề. Dọn tag trắng, trùng và thiếu tên. Nếu mảng tag bài hát rỗng sau khi dọn, dùng tag nghệ sĩ. Nếu một event được crawl nhiều lần, ưu tiên bản có tag bài, kế đến bản có tag nghệ sĩ. Khi cùng nguồn, chọn danh sách có nhiều tag hơn, rồi theo thứ tự tên để kết quả lặp lại được. Ghép metadata vào lượt nghe bằng `event_id`.

Mỗi lượt đóng góp tổng trọng số 1. Ví dụ, một bài có 2 tag đóng góp 0,5 cho mỗi tag. Lượt thiếu cả hai nguồn vào nhóm “(thiếu tag)”. Trong mỗi giờ, chia trọng số cho số lượt của từng user tại giờ đó, rồi lấy trung bình đều giữa **các user có nghe trong giờ**. Vì thế tổng tỷ lệ của tất cả tag trong một giờ là 100%. Bảng hiển thị mọi tag xuất hiện trong từng giờ UTC, sắp theo tỷ lệ giảm dần.

In [8]:
def clean_tags(group_path):
    # Khai báo kiểu phần tử để cả lô toàn mảng [] vẫn đọc được trường name.
    tag_group = F.from_json(F.to_json(F.col(group_path)), "tag ARRAY<STRUCT<name:STRING>>")
    names = F.transform(tag_group["tag"], lambda t: F.lower(F.trim(t["name"])))
    cleaned = F.sort_array(F.array_distinct(F.filter(names, lambda x: x.isNotNull() & (x != ""))))
    return F.when(F.size(cleaned) > 0, cleaned)

tag_candidates = (
    eligible.select("event_id", clean_tags("track_metadata.toptags").alias("track_tags"),
                    clean_tags("artist_metadata.tags").alias("artist_tags"))
    .withColumn("tags", F.coalesce("track_tags", "artist_tags"))
    .withColumn("priority", F.when(F.col("track_tags").isNotNull(), 2)
                .when(F.col("artist_tags").isNotNull(), 1).otherwise(0))
)
event_tags = (
    tag_candidates.groupBy("event_id")
    .agg(F.max(F.struct("priority", F.size("tags").alias("n"), "tags")).alias("best"))
    .select("event_id", F.col("best.tags").alias("tags"),
            F.when(F.col("best.priority") == 2, "track")
             .when(F.col("best.priority") == 1, "artist").otherwise("missing").alias("tag_source"))
)
tagged = plays.select("event_id", "user", "gio_utc").join(event_tags, "event_id")
show_all(tagged.groupBy("tag_source").count().orderBy("tag_source"))

user_hour = Window.partitionBy("user", "gio_utc")
weighted = (
    tagged.withColumn("luot_user_gio", F.count("*").over(user_hour))
    .withColumn("labels", F.coalesce("tags", F.array(F.lit("(thiếu tag)"))))
    .withColumn("weight", 1.0 / (F.size("labels") * F.col("luot_user_gio")))
    .withColumn("tag", F.explode("labels"))
)
by_hour = (
    plays.groupBy("gio_utc")
    .agg(F.countDistinct("user").alias("so_user_co_nghe"))
)
hour_tags = (
    weighted.groupBy("gio_utc", "tag").agg(F.sum("weight").alias("sum_share"))
    .join(by_hour.select("gio_utc", "so_user_co_nghe"), "gio_utc")
    .withColumn("share_pct", 100 * F.col("sum_share") / F.col("so_user_co_nghe"))
    .drop("sum_share")
)
show_all(hour_tags.orderBy("gio_utc", F.desc("share_pct"), "tag"))

tag_source	count
artist	205864
missing	25365
track	33775
gio_utc	tag	so_user_co_nghe	share_pct
0	(thiếu tag)	449	8.874473933819948
0	electronic	449	3.4838807598676165
0	rock	449	3.4241491417351098
0	hip-hop	449	3.255826606062672
0	rap	449	3.1995283456892447
0	hip hop	449	2.6304157682915417
0	alternative	449	2.3919343376504116
0	indie	449	2.305222486387114
0	pop	449	2.00314459967079
0	80s	449	1.9320329089003356
0	rnb	449	1.701752112492228
0	new wave	449	1.6968797721076263
0	kanye west	449	1.5155363056512536
0	female vocalists	449	1.4378743347853766
0	synthpop	449	1.3843263149788505
0	alternative rock	449	1.0935477882470286
0	indie rock	449	1.0660106224944967
0	experimental	449	1.0112057073111123
0	trap	449	0.9558820317276813
0	metal	449	0.9151209079951235
0	ambient	449	0.8765393333859932
0	post-punk	449	0.8277176429362625
0	synth pop	449	0.8127133882056561
0	singer-songwriter	449	0.7910709462716506
0	soul	449	0.786166290599374
0	jazz	449	0.743299819724884
0	classic rock	449	0.7152975478

**Khi nối với bảng gợi ý**. Nối đặc trưng thói quen ở mức user bằng `user`. Với gợi ý nghệ sĩ của user, dùng `user + artist_key`. Với album, dùng `user + album_key` (định danh album gồm nghệ sĩ và album). Với cạnh similar, dùng `artist_key`. Dùng cùng quy tắc tạo khóa của bảng gợi ý. `uts` dùng để sắp xếp, lọc, tạo nhóm thời gian và đặt mốc train/test. Nếu dùng các tỷ lệ này làm đặc trưng huấn luyện, hãy lọc lượt nghe trước mốc test rồi tính lại các tỷ lệ trên tập train.

## Xem toàn bộ bản ghi Bronze
Cell này in JSONL gốc của tất cả file trong lần đọc `raw`, mỗi dòng là một lượt nghe. Chạy riêng cell này khi muốn xem dữ liệu thô. Với bucket lớn, VS Code có thể thu gọn hoặc giới hạn phần output hiển thị.

In [ ]:
from contextlib import closing

def iter_bronze_jsonl(paths):
    """Đọc tuần tự từng dòng UTF-8 của các file .jsonl.gz trong cùng bucket."""
    if not paths:
        return
    jvm = spark.sparkContext._jvm
    PathJ = jvm.org.apache.hadoop.fs.Path
    conf = spark.sparkContext._jsc.hadoopConfiguration()
    fs = jvm.org.apache.hadoop.fs.FileSystem.newInstance(PathJ(paths[0]).toUri(), conf)
    with closing(fs):
        for path in paths:
            with closing(fs.open(PathJ(path))) as source:
                gzip_stream = jvm.java.util.zip.GZIPInputStream(source)
                with closing(gzip_stream):
                    decoder = jvm.java.io.InputStreamReader(gzip_stream, "UTF-8")
                    with closing(jvm.java.io.BufferedReader(decoder)) as reader:
                        yield from iter(reader.readLine, None)

with closing(iter_bronze_jsonl(sorted(raw.inputFiles()))) as lines:
    for line in lines:
        print(line)